In [ ]:
# from imdclient.IMD import IMDReader
import MDAnalysis as mda
import numpy as np
from graph_utils import live_plot, live_plot_multi

In [ ]:
dt = 0.002 # ps
unitConv = 1.6022e-19 * 1.0e9 / 10e-12 # (e*A)/ps to nA
ionText = "name POT CLA"

In [ ]:
#%matplotlib widget
NAMD_TOPOL = "../md/T3_MNN.psf"

plot = live_plot(
    title="Ion Current vs. Time", 
    xaxLabel="time (ps)", 
    yaxLabel="current (nA)",
    dataLabel = "ion current",
    mode = 'append'
)
# fixed y-axis for visualization
plot["ax"].set_ylim(-25,25)

u = mda.Universe(NAMD_TOPOL, "imd://10.206.72.141:8888", buffersize = 1024*1024*1024)
ions = u.select_atoms(ionText)
charges = ions.charges
dims = u.dimensions
zdist = dims[2]

i = 0
for t in u.trajectory:
    time = i * dt
    ionPos = ions.positions
    if i == 0:
        ionRef = ionPos
        current = 0.0
    else:
        deltas = ionPos[:,2]-ionRef[:,2]
        # correct for PBC jumps
        deltas[deltas>zdist/2.] -= zdist
        deltas[deltas<-zdist/2.] += zdist
        Qd = deltas*charges
        current = np.sum(Qd)/zdist/dt*unitConv
        ionRef = ionPos
        plot['update'](time, current)
    i = i + 1

In [ ]:
#%matplotlib widget
NAMD_TOPOL = "../md/T3_MNN.psf"

plot = live_plot_multi(
    title="Ion Current vs. Time", 
    xaxLabel="time (ps)", 
    yaxLabel="current (nA)",
    dataLabels = ["cation current", "anion current"],
    mode = 'append'
)
# fixed y-axis for visualization
plot["ax"].set_ylim(-25,25)

u = mda.Universe(NAMD_TOPOL, "imd://10.206.72.141:8888", buffersize = 1024*1024*1024)
cations = u.select_atoms("name POT")
anions = u.select_atoms("name CLA")
cationQ = cations.charges
anionQ = anions.charges
dims = u.dimensions
zdist = dims[2]

i = 0
for t in u.trajectory:
    time = i * dt
    cationPos = cations.positions
    anionPos = anions.positions
    if i == 0:
        cationRef = cationPos
        anionRef = anionPos
        cationCurrent = 0.0
        anionCurrent = 0.0
    else:
        cationDeltas = cationPos[:,2]-cationRef[:,2]
        anionDeltas = anionPos[:,2]-anionRef[:,2]
        # correct for PBC jumps
        cationDeltas[cationDeltas>zdist/2.] -= zdist
        cationDeltas[cationDeltas<-zdist/2.] += zdist
        anionDeltas[anionDeltas>zdist/2.] -= zdist
        anionDeltas[anionDeltas<-zdist/2.] += zdist
        
        cationQd = cationDeltas*cationQ
        anionQd = anionDeltas*anionQ
        cationCurrent = np.sum(cationQd)/zdist/dt*unitConv
        anionCurrent = np.sum(anionQd)/zdist/dt*unitConv
        cationRef = cationPos
        anionRef = anionPos
        plot['update'](time, [cationCurrent, anionCurrent])
    i = i + 1